In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import geopandas as gpd
import plotly.express as px 
import plotly.io as pio
import plotly.subplots as sp
import plotly.graph_objects as go
from tqdm import tqdm
import calendar
pio.renderers.default = "notebook+browser"




In [3]:
#apply dtype mappings while reading the csv
#also apply format date time columns
#also make Police District Number as categorical    

# Load the CSV without applying dtype mappings initially
cd = pd.read_csv('./data/Crime_Dataset.csv', parse_dates=['Dispatch Date / Time', 'Start_Date_Time', 'End_Date_Time'])
cd.info()





/var/folders/s0/0t4b0q1x7rj9l6hs1khgqxsr0000gn/T/ipykernel_20532/2333865301.py:6: DtypeWarning: Columns (1,18) have mixed types. Specify dtype option on import or set low_memory=False.
  cd = pd.read_csv('./data/Crime_Dataset.csv', parse_dates=['Dispatch Date / Time', 'Start_Date_Time', 'End_Date_Time'])
/var/folders/s0/0t4b0q1x7rj9l6hs1khgqxsr0000gn/T/ipykernel_20532/2333865301.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cd = pd.read_csv('./data/Crime_Dataset.csv', parse_dates=['Dispatch Date / Time', 'Start_Date_Time', 'End_Date_Time'])
/var/folders/s0/0t4b0q1x7rj9l6hs1khgqxsr0000gn/T/ipykernel_20532/2333865301.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cd = pd.read_csv('./data/Crime_Dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306094 entries, 0 to 306093
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Incident ID             306094 non-null  int64         
 1   Offence Code            306094 non-null  object        
 2   CR Number               306094 non-null  int64         
 3   Dispatch Date / Time    257065 non-null  datetime64[ns]
 4   NIBRS Code              306094 non-null  object        
 5   Victims                 306094 non-null  int64         
 6   Crime Name1             305822 non-null  object        
 7   Crime Name2             305822 non-null  object        
 8   Crime Name3             305822 non-null  object        
 9   Police District Name    306000 non-null  object        
 10  Block Address           279888 non-null  object        
 11  City                    304818 non-null  object        
 12  State                   306094

# Fixes

# Drop the duplicate entries base on Incident ID

In [ ]:
#DROPS
#drop dupelicated entries based on Incident ID
cd[cd.duplicated(subset=['Incident ID'])]
cd = cd.drop_duplicates(subset=['Incident ID'])


<class 'pandas.core.frame.DataFrame'>
Index: 280928 entries, 0 to 306093
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Incident ID             280928 non-null  int64         
 1   Offence Code            280928 non-null  object        
 2   CR Number               280928 non-null  int64         
 3   Dispatch Date / Time    235939 non-null  datetime64[ns]
 4   NIBRS Code              280928 non-null  object        
 5   Victims                 280928 non-null  int64         
 6   Crime Name1             280690 non-null  object        
 7   Crime Name2             280690 non-null  object        
 8   Crime Name3             280690 non-null  object        
 9   Police District Name    280843 non-null  object        
 10  Block Address           257957 non-null  object        
 11  City                    279744 non-null  object        
 12  State                   280928 non-

# Fill NaN vluese in integer columns with 0 before  

In [ ]:
for col, dtype in dtype_mappings.items():
	if dtype == 'int64' and col in cd.columns:
		cd[col] = cd[col].fillna(0).astype('int64')
	elif col in cd.columns:
		cd[col] = cd[col].astype(dtype)

# Fix Offence Code Column

In [ ]:
cd['Offence Code'] = cd['Offence Code'].replace({'FTAS': '9998', 'NSUP': '9999'})
cd['Offence Code']= cd['Offence Code'].astype('int64')
# Copy Start_Date_Time to Dispatch Date / Time if Dispatch Date / Time is NaT
cd['Dispatch Date / Time'] = cd['Dispatch Date / Time'].fillna(cd['Start_Date_Time'])

# Police District Number to fix and change to categorical


In [ ]:
def fix_police_districts(df, num_col="Police District Number", name_col="Police District Name"):
    # Normalize code to D form: e.g., 1.0D -> 1D
    norm = df[num_col].astype(str).str.replace(".0D", "D", regex=False)

    # Map: normalized code -> known name (first non-null per code)
    name_map = (pd.DataFrame({"_norm": norm, name_col: df[name_col]})
                .dropna(subset=[name_col])
                .drop_duplicates("_norm")
                .set_index("_norm")[name_col])

    # Fill NaNs for rows like *.0D from the mapped name
    needs_fill = df[name_col].isna() & df[num_col].astype(str).str.endswith(".0D")
    df.loc[needs_fill, name_col] = norm[needs_fill].map(name_map)

    # Finally, normalize all codes to the D form
    df[num_col] = norm
    return df

cd = fix_police_districts(cd)
cd['Police District Name'] = cd['Police District Name'].replace({'CITY OF TAKOMA PARK': 'TAKOMA PARK'})


# Block Address Construction form Address Number, Street Name, Street Suffix, Street Prefix and Street Type

Entry with NaN or empty Block Address will be filled by copying Address Number and Street Name 

Format Address Number + Street Prefix + Street Name + Street Suffix +Street Type 

If Address Number is 0 (or missing after fill), mark as Unknown

In [ ]:
mask_missing = cd['Block Address'].isna() | (cd['Block Address'].str.strip() == '')
addr = cd['Address Number'].fillna(0).astype(int)   # ensure int
street = cd['Street Name'].fillna('Unknown Street')
suffix = cd['Street Suffix'].fillna('Unknown Suffix')
street_prefix = cd['Street Prefix'].fillna('')
street_type = cd['Street Type'].fillna('')

# Instead of creating a 100-block range (e.g. 100-199), copy the address number
cd.loc[mask_missing & (addr == 0), 'Block Address'] = 'Unknown Block of ' + street_prefix + ' ' + street + ' ' + suffix + ' ' + street_type
cd.loc[mask_missing & (addr != 0), 'Block Address'] = addr.astype(str) + ' ' + street_prefix + ' ' + street + ' ' + suffix + ' ' + street_type



# Latitude and Logitude Construction using geolocator package



In [ ]:
#Api Calls Setup
geolocator = Nominatim(user_agent="geoapi")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)
pd.set_option('display.max_columns', None)

In [ ]:
#Get City from Latitude and Longitude using geopy
def get_city(lat, lon):
    try:
        location = reverse((lat, lon), language="en")
        if location and "address" in location.raw:
            addr = location.raw["address"]
            return (
                addr.get("city")
                or addr.get("town")
                or addr.get("village")
                or addr.get("county")
            )
    except Exception:
        return None
    return None
tqdm.pandas()
# preview = cd.apply(lambda r: get_city(r["Latitude"], r["Longitude"]), axis=1)
# print(preview)
mask = cd["City"].isna() & cd["Latitude"].notna() & cd["Longitude"].notna()
cd.loc[mask, "City"] = cd[mask].progress_apply(lambda r: get_city(r["Latitude"], r["Longitude"]), axis=1)


100%|██████████| 1183/1183 [19:42<00:00,  1.00it/s]


# Getting Missing Lat and Lon from Available Zip Codes

In [ ]:

# -----------------------------
# Step 0: Load your dataset
# -----------------------------
# Replace with your actual file path
cd = pd.read_csv("./data/Crime_Dataset_Cleaned.csv")

# Ensure Latitude and Longitude are numeric
cd['Latitude'] = pd.to_numeric(cd['Latitude'], errors='coerce')
cd['Longitude'] = pd.to_numeric(cd['Longitude'], errors='coerce')

# -----------------------------
# Step 1: Compute average lat/lon per ZIP
# -----------------------------
# Use rows that already have valid coordinates
zip_avg = (
    cd[(cd['Latitude'].notna()) & (cd['Latitude'] != 0)]
    .groupby('Zip Code')[['Latitude', 'Longitude']]
    .mean()
    .reset_index()
)

# -----------------------------
# Step 2: Fill missing lat/lon using ZIP averages
# -----------------------------
# Filter rows with missing or zero lat/lon
# Filter rows with missing or zero lat/lon (keep index)
missing_idx = cd[(cd['Latitude'] == 0) | (cd['Longitude'] == 0)].index

# Merge ZIP averages (same as before)
missing_lat_lon = cd.loc[missing_idx].copy()
missing_lat_lon = missing_lat_lon.merge(
    zip_avg,
    on='Zip Code',
    how='left',
    suffixes=('', '_zipavg')
)

# Fill missing lat/lon
missing_lat_lon['Latitude'] = missing_lat_lon['Latitude'].replace(0, np.nan).fillna(missing_lat_lon['Latitude_zipavg'])
missing_lat_lon['Longitude'] = missing_lat_lon['Longitude'].replace(0, np.nan).fillna(missing_lat_lon['Longitude_zipavg'])

# Update the original DataFrame using the original indices
cd.loc[missing_idx, 'Latitude'] = missing_lat_lon['Latitude']
cd.loc[missing_idx, 'Longitude'] = missing_lat_lon['Longitude']

# Verify
print(cd[['Latitude', 'Longitude']].isna().sum())
print(cd[['Latitude', 'Longitude']].sample(25))


cd.to_csv('./data/Crime_Dataset_Cleaned.csv', index=False)

# Call the Clean data set with the data types applied and parse dates

In [3]:
dtype_mappings = {
    'Incident ID': 'Int64',
    'Offense Code': 'Int64',
    'CR Number': 'Int64',
    'Dispatch Date / Time': 'string',
    'NIBRS Code': 'string',
    'Victims': 'Int64',
    'Crime Name1': 'category',
    'Crime Name2': 'string',
    'Crime Name3': 'string',
    'Police District Name': 'string',
    'Block Address': 'string',
    'City': 'string',
    'State': 'string',
    'Zip Code': 'Int64',
    'Agency': 'string',
    'Place':'string',
    'Sector':'string',
    'Beat':'string',
    'PRA':'string',
    'Address Number':'Int64', 
    'Street Prefix':'string',
    'Street Name':'string',
    'Street Suffix':'string',
    'Street Type':'string',
    'Start_Date_Time':'string',
    'End_Date_Time':'string',
    'Latitude':'float64',
    'Longitude':'float64',
    'Police District Number': 'string', 
    'Location': 'string'
}

# Data Preparation for analysis

In [157]:
cd =pd.read_csv('./data/Crime_Dataset_Cleaned.csv', dtype=dtype_mappings, parse_dates=['Dispatch Date / Time', 'Start_Date_Time', 'End_Date_Time'])
cd['Crime Name1'] = cd['Crime Name1'].astype('category')


cd['Year'] = cd['Dispatch Date / Time'].dt.year
cd['Month'] = cd['Dispatch Date / Time'].dt.month
cd['Season'] = cd['Month'] % 12 // 3 + 1
cd['City'] = cd['City'].str.upper()
season_map = {
    1: 'Winter',
    2: 'Spring',
    3: 'Summer',
    4: 'Autumn'
}

cd['SeasonName'] = cd['Season'].map(season_map)



def categorize_place(place):
    p = str(place).lower()

    # Street / Roadway
    if any(x in p for x in [
        "street", "road", "bus stop", "vehicle", "alley",
        "commercial", "pedestrian tunnel", "terminal", "rest area", "dock"
    ]):
        return "Street / Roadway"

    # Residential
    if "residence" in p or "apartment" in p or "condo" in p or "townhouse" in p \
       or "driveway" in p or "yard" in p or "shed" in p or "carport" in p \
       or "garage" in p or "mobile home" in p or "nursing home" in p:
        return "Residential"

    # Parking Areas
    if "parking" in p or "park & ride" in p or "parking garage" in p:
        return "Parking Area"

    # Retail / Commercial
    if any(x in p for x in [
        "retail", "store", "mall", "restaurant", "bar", "night club",
        "gas station", "convenience store", "liquor", "auto dealership",
        "repair", "commercial", "hardware", "salon", "spa", "jewelry",
        "clothing", "appliances", "electronics", "sporting goods",
        "beauty", "barber", "dry cleaner", "video store",
        "bank", "atm", "checking", "pawn", "doctor", "dentist",
        "hospital", "laundromat", "community center", "jail",
        "rental storage"
    ]):
        return "Commercial / Retail"

    # School / Government
    if any(x in p for x in [
        "school", "college", "government", "library", "church",
        "synagogue", "temple", "recreation center"
    ]):
        return "School / Government"

    # Outdoor / Open Areas
    if any(x in p for x in [
        "park", "open space", "wooded", "golf", "pool",
        "lake", "waterway", "camp", "amusement", "stadium",
        "fairgrounds", "farm", "construction", "industrial",
        "abandoned"
    ]):
        return "Outdoor / Open Area"

    # Other
    return "Other / Unknown"
cd['Place_Category'] = cd['Place'].apply(categorize_place)

# Prepare data for future analysis
df = cd.copy()
df['Start_Date_Time'] = pd.to_datetime(df['Start_Date_Time'], errors='coerce')
df = df.dropna(subset=['Start_Date_Time'])
df['Date'] = df['Start_Date_Time'].dt.date
df['Hour'] = df['Start_Date_Time'].dt.hour
df['Day_Name'] = df['Start_Date_Time'].dt.day_name()
df['Day_Type'] = df['Start_Date_Time'].dt.dayofweek.isin([5, 6]).map({True: 'Weekend', False: 'Weekday'})

df['Start_Date_Time'] = pd.to_datetime(df['Start_Date_Time'], errors='coerce') 
df = df.dropna(subset=['Start_Date_Time']) 
df['Hour'] = df['Start_Date_Time'].dt.hour
df['Day_Name'] = df['Start_Date_Time'].dt.day_name()


#Classify risk categories
risk_mapping = {
    'Simple Assault': 'Assault / Physical Violence',
    'Aggravated Assault': 'Assault / Physical Violence',
    'Forcible Rape': 'Sexual Violence',
    'Fondling': 'Sexual Violence',
    'Forcible Fondling': 'Sexual Violence',
    'Forcible Sodomy': 'Sexual Violence',
    'Intimidation': 'Stalking & Harassment',
    'Sexual Assault With An Object': 'Sexual Violence',
    'All Other Offenses': 'Other',
    'Murder and Nonnegligent Manslaughter': 'Homicide',
    'Kidnapping/Abduction': 'Homicide / Abduction',
    'Statuory Rape': 'Sexual Violence',
    'Human Trafficking, Commercial Sex Acts': 'Human Trafficking',
    'Justifiable Homicide': 'Homicide',
    'Negligent Manslaughter': 'Homicide',
    'Human Trafficking, Involuntary Servitude': 'Human Trafficking'
}

df['Risk_Category'] = df['Crime Name2'].map(risk_mapping)
df_risk = df[df['Risk_Category'].notnull()]
df_risk['Place_Type'] = df_risk['Place'].apply(categorize_place)

#Prepare days and area for visualizations
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'] 
top_districts = ['MONTGOMERY VILLAGE', 'WHEATON', 'SILVER SPRING', 'ROCKVILLE', 'GERMANTOWN'] 
heatmap_data = df_risk[df_risk['Police District Name'] == 'MONTGOMERY VILLAGE'].groupby(['Day_Name', 'Hour']).size().unstack(fill_value=0).reindex(days_order) 
loc_counts = df_risk[df_risk['Police District Name'].isin(top_districts)].groupby(['Police District Name', 'Place_Type']).size().reset_index(name='Count') 
comp_counts = df_risk[df_risk['Police District Name'].isin(top_districts)].groupby(['Police District Name', 'Risk_Category']).size().unstack(fill_value=0)





C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\26511383.py:1: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\26511383.py:110: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [101]:
cd[cd['Crime Name1']=='Crime Against Person']['Crime Name2'].value_counts()

Crime Name2
Simple Assault                              18701
Aggravated Assault                           3752
Forcible Rape                                1194
Fondling                                     1025
Forcible Fondling                             508
Forcible Sodomy                               397
Intimidation                                  388
Sexual Assault With An Object                 246
All Other Offenses                            120
Murder and Nonnegligent Manslaughter          105
Kidnapping/Abduction                           57
Statuory Rape                                  31
Human Trafficking, Commercial Sex Acts         15
Justifiable Homicide                           14
Negligent Manslaughter                          2
Human Trafficking, Involuntary Servitude        2
Name: count, dtype: Int64

In [5]:
cd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280627 entries, 0 to 280626
Data columns (total 33 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Incident ID             280627 non-null  Int64         
 1   Offence Code            280627 non-null  int64         
 2   CR Number               280627 non-null  Int64         
 3   Dispatch Date / Time    280627 non-null  datetime64[ns]
 4   NIBRS Code              280627 non-null  string        
 5   Victims                 280627 non-null  Int64         
 6   Crime Name1             280627 non-null  category      
 7   Crime Name2             280627 non-null  string        
 8   Crime Name3             280627 non-null  string        
 9   Police District Name    280627 non-null  string        
 10  Block Address           280627 non-null  string        
 11  City                    280627 non-null  string        
 12  State                   280627

# Q1. Crime counts for each tpye. And what kind of crime is the most prevalent in each district and in which place crime mostly occurs? !!Has 2 Vis

In [159]:
crime_counts = cd.groupby(['Police District Name', 'Crime Name1']).size().reset_index(name='Count')

crime_counts['Percentage'] = crime_counts.groupby('Police District Name')['Count'].transform(lambda x: (x / x.sum()) * 100)
crime_counts['Label'] = crime_counts.apply(lambda row: f"{row['Crime Name1']} – {row['Percentage']:.1f}%", axis=1)

crime_counts = crime_counts[crime_counts['Police District Name'] != 'OTHER']
place_counts = cd['Place_Category'].value_counts().reset_index()
place_counts.columns = ['Place_Category', 'Count']

color_map = {
    "Street / Roadway": "#66b3ff",
    "Residential": "#ff9999",
    "Parking Area": "#99ff99",
    "Commercial / Retail": "#ffcc66",
    "School / Government": "#c2c2f0",
    "Outdoor / Open Area": "#ffb3e6",
    "Other / Unknown": "#cccccc"
}

fig = sp.make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "treemap"}, {"type": "domain"}]],   # domain = pie
    column_widths=[0.65, 0.35],
    subplot_titles=[
        "Crime Distribution by Police District and Type", 
        "Overall Crime Breakdown (Pie)"
    ]
)


fig1 = px.treemap(
    crime_counts,
    path=['Police District Name', 'Label'],  
    values='Count',
    color='Police District Name',
    color_discrete_sequence=px.colors.qualitative.Set3,
)
for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

fig2 = px.pie(
    place_counts,
    names='Place_Category',
    values='Count',
    color='Place_Category',
    color_discrete_map=color_map
)
for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)


fig.update_layout(
    template='plotly_white',
    width=1400,
    height=1000,
)

fig.show()

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\3284255183.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![tremap for crime](./images/Q1.png "a title")

# Q2. What is the Monthly Crime Rate in Each Region? !!Has 2 Vis
We visualise the figure of crime in each city in the region for each month using heatmap.

In [73]:
# Top 5 crime types
top_crimes = cd['Crime Name1'].value_counts().nlargest(5).index
df_top = cd[cd['Crime Name1'].isin(top_crimes)]

# Group by Month and Crime Type
crime_month_type = df_top.groupby(['Month','Crime Name1']).size().reset_index(name='Count')

# Total crimes per month
total_crimes_month = crime_month_type.groupby('Month')['Count'].sum().reset_index()

# Find the month with highest total crime
max_month = total_crimes_month.loc[total_crimes_month['Count'].idxmax(), 'Month']
max_total = total_crimes_month['Count'].max()

fig = sp.make_subplots(
    rows=2, cols=1,
    specs=[
    [{"type": "scatter"}],
    [{"type": "heatmap"}]
    ],
    subplot_titles=[
        "Crime Types by Month",
        "Crime Types by Season"
    ]
)

#fig1
fig.add_trace(go.Scatter(
    x=crime_month_type['Month'],
    y=crime_month_type['Crime Name1'],
    mode='markers+text',
    marker=dict(
        size=crime_month_type['Count'],   # bubble size
        sizemode='area',
        sizeref=2.*crime_month_type['Count'].max()/(100**2),  
        sizemin=10,
        color=crime_month_type['Crime Name1'].astype('category').cat.codes,
        
    ),
    text=crime_month_type['Count'],
    textposition='middle center',
    textfont=dict(color='black', size=10),
    hovertemplate='<b>%{y}</b><br>Month: %{x}<br>Count: %{text}<extra></extra>',
    showlegend=False
), row=1, col=1)


for _, row in total_crimes_month.iterrows():
    month = row['Month']
    total = int(row['Count'])

    y_pos = len(top_crimes) + 0.6

    if month == max_month:
        fig.add_annotation(
            x=month, y=y_pos, text=f"<b>{total}</b>",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="darkred",
            yshift=15,
            font=dict(color="darkred", size=14)
        )
        fig.add_annotation(
            x=month, y=y_pos+0.2,
            text="<b>Highest</b>",
            showarrow=False,
            font=dict(color='darkred', size=12)
        )
    else:
        fig.add_annotation(
            x=month, y=y_pos,
            text=f"<b>{total}</b>",
            showarrow=False,
            font=dict(color='black', size=12)
        )


# Axis settings for left plot
fig.update_xaxes(
    title='Month',
    tickmode='array',
    tickvals=list(range(1, 13)),
    row=1, col=1
)
fig.update_yaxes(
    title='Crime Type',
    categoryorder='array',
    categoryarray=top_crimes,
    row=1, col=1
)

fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='lightgray', row=1, col=1)
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='lightgray', row=1, col=1)


#fig2
crime_by_season_type = cd.groupby(['SeasonName', 'Crime Name1']).size().reset_index(name='Count')

crime_pivot = crime_by_season_type.pivot(
    index='Crime Name1',
    columns='SeasonName',
    values='Count'
).fillna(0)

crime_pivot = crime_pivot[['Winter', 'Spring', 'Summer', 'Autumn']]

crime_pivot['Total'] = crime_pivot.sum(axis=1)

# Order by descending total
crime_pivot = crime_pivot.sort_values('Total', ascending=True)

# Drop the temporary Total column for plotting
crime_pivot = crime_pivot.drop(columns='Total')

heatmap = go.Heatmap(
    z=crime_pivot.values,
    x=crime_pivot.columns,
    y=crime_pivot.index,
    text=crime_pivot.values, 
    showscale=True,
    texttemplate="%{text}",     
    colorscale='Reds',
    colorbar=dict(
        len=0.4,
        y=0.19,
    ),
)

fig.add_trace(heatmap, row=2, col=1)

fig.update_xaxes(title="Season", row=2, col=1)
fig.update_yaxes(title="Crime Type", row=2, col=1)


fig.update_layout(
        height=1200,
    template="plotly_white"
)
fig.show()


C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\613351860.py:6: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\613351860.py:99: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![Crime Type Chart](./images/Q2.png "a title")

# Q3. How each type of Crime distribute over the year and place they occur? !!Has 2 Vis
First we look generally across every region so in this section we analyse how crime works over the year. 

In [97]:


yearly_type_counts = cd.groupby(['Year', 'Crime Name1']).size().reset_index(name='Count')

place_counts = cd['Place_Category'].value_counts().reset_index()
place_counts.columns = ['Place_Category', 'Count']

color_map = {
    "Street / Roadway": "#66b3ff",
    "Residential": "#ff9999",
    "Parking Area": "#99ff99",
    "Commercial / Retail": "#ffcc66",
    "School / Government": "#c2c2f0",
    "Outdoor / Open Area": "#ffb3e6",
    "Other / Unknown": "#cccccc"
}

# adjust height and width for better visibility
fig = sp.make_subplots(
    rows=2, cols=3,
    specs=[
        [{"type": "xy", "colspan": 3}, None, None],
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}]
    ],
    subplot_titles=[
        "Yearly Crime Counts by Crime Type", 
        "Crime Against Property by Place Category", 
        "Crime Against Society by Place Category",
        "Crime Against Person by Place Category",
        "",
        ""
    ]
)

fig1 = px.bar(
    yearly_type_counts,
    x='Year',
    y='Count',
    color='Crime Name1',
    barmode='group',
    title='Yearly Crime Counts by Crime Type',
    color_discrete_sequence=px.colors.sequential.Viridis_r
)

for trace in fig1.data:
    trace.legendgroup = "group1"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=1)

place_counts_property = cd[cd['Crime Name1'].str.contains('Property', na=False)]['Place_Category'].value_counts().reset_index()
place_counts_property.columns = ['Place_Category', 'Count']

fig2 = px.pie(
    place_counts_property,
    names='Place_Category',
    values='Count',
    color='Place_Category',
    color_discrete_map=color_map
)
for trace in fig2.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=2, col=1)

place_counts_society = cd[cd['Crime Name1'].str.contains('Society', na=False)]['Place_Category'].value_counts().reset_index()
place_counts_society.columns = ['Place_Category', 'Count']

fig3 = px.pie(
    place_counts_society,
    names='Place_Category',
    values='Count',
    color='Place_Category',
    color_discrete_map=color_map
)
for trace in fig3.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=2, col=3)

place_counts_person = cd[cd['Crime Name1'].str.contains('Person', na=False)]['Place_Category'].value_counts().reset_index()
place_counts_person.columns = ['Place_Category', 'Count']
fig4 = px.pie(
    place_counts_person,
    names='Place_Category',
    values='Count',
    color='Place_Category',
    color_discrete_map=color_map,
)
for trace in fig4.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=2, col=2)

fig.update_layout(
    template='plotly_white',
    legend=dict(
        y=.96,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    legend2=dict(   
        y=.25,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    height=900,
    width=1600
)
for i, trace in enumerate(fig.data):
    if trace.legendgroup == "group2":
        trace.legend = "legend2"


fig.update_yaxes(title_text="Year", row=1, col=1)
fig.update_yaxes(title_text="Number of Crimes", row=1, col=1)

fig.show()


C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\479219052.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![Crime Type Chart](./images/Q3.png "a title")

# Q4. Look into seasonal crimes, does car burglary happen more in winter due to earlier darkness or the rise of usage of marijuana during summer times due to festivals and stuff? !!Has 2 Vis
First, when we looked into the crime data, we found that Crime against property is the highest. So we decided to look into it more deeply. From detailed analysis, we can see that Vehicle related crime are the most common. It is around 30% of all crimes in Crime against Property. So, we decided to research why vehicle related crime is more than any other sub group. When we relate it to the papers online we found that it is due to Marijuana usage during festival time. 


In [ ]:
# For property crimes by type
property_crimes = cd[cd['Crime Name1'] == 'Crime Against Property']
property_crime_counts = property_crimes['Crime Name2'].value_counts().reset_index()
property_crime_counts.columns = ['Crime Name2', 'Count']

# Define distinct colors for highlight types
highlight = [
    'Motor Vehicle Theft',
    'Theft of Motor Vehicle Parts or Accessories',
    'Theft From Motor Vehicle'
]
highlight_colors = {
    'Motor Vehicle Theft': "#f881d0",       # red
    'Theft of Motor Vehicle Parts or Accessories': "#78f174",   # green
    'Theft From Motor Vehicle': "#29e9d9",  # blue
}

# Neutral color for all other crime types
default_color = "#E7E5E5"
color_map = {
    row['Crime Name2']: highlight_colors.get(row['Crime Name2'], default_color)
    for _, row in property_crime_counts.iterrows()
}
seasonal_type_counts = cd.groupby(['Season', 'Crime Name2']).size().reset_index(name='Count')
seasonal_type_counts = seasonal_type_counts[seasonal_type_counts['Crime Name2'].isin(['Motor Vehicle Theft', 'Theft of Motor Vehicle Parts or Accessories', 'Theft From Motor Vehicle'])]
seasonal_type_counts['Season'] = seasonal_type_counts['Season'].map({1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'})

# Filter top 5 crime types
top_crimes = cd['Crime Name1'].value_counts().nlargest(5).index
df_top = cd[cd['Crime Name1'].isin(top_crimes)]

# Group by Month and Crime Type
crime_month_type = df_top.groupby(['Month','Crime Name1']).size().reset_index(name='Count')

# Total crimes per month
total_crimes_month = crime_month_type.groupby('Month')['Count'].sum().reset_index()

# Find the month with highest total crime
max_month = total_crimes_month.loc[total_crimes_month['Count'].idxmax(), 'Month']
max_total = total_crimes_month['Count'].max()


fig = sp.make_subplots(
    rows=1, cols=2,
    specs=[
        [{"type": "domain"}, {"type": "xy"}]  
    ],
    shared_xaxes=False,
    horizontal_spacing=.5,
    subplot_titles=[
        "Distribution of Property Crimes by Type",
        "Seasonal Crime Counts"
    ]
)

fig1 = px.pie(
    property_crime_counts,
    names='Crime Name2',
    values='Count',
    color='Crime Name2',
    color_discrete_map=color_map
)
for trace in fig1.data:
    trace.legendgroup = "group1"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=1)

fig2 = px.bar(
    seasonal_type_counts,
    x='Season',
    y='Count',
    color='Crime Name2',
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
for trace in fig2.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=2)


fig.update_layout(
    template='plotly_white',
    height=600,
    legend=dict(
        x=0.38,
        y=.96,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    legend2=dict(   
        x=0.85,
        y=.95,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
for i, trace in enumerate(fig.data):
    if trace.legendgroup == "group2":
        trace.legend = "legend2"
        
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_xaxes(title_text="Seasons", row=1, col=1)
fig.update_yaxes(title_text="Number of Crimes", row=1, col=1)

fig.show()


C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\4125839214.py:33: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![Crime Type Chart](./images/Q4.png "Q4")

# Q5. Which City is very unsafe for car owners? !!Has 2 Vis
Since we are onto the subject we look do city analysis for Vehicle related crime for each city.

In [ ]:
# First one is about vehicle related crimes over the years by city
# Second one is about where it happens the most Place for each type of risk category

vehicle_related_types = [
    'Motor Vehicle Theft',
    'Theft of Motor Vehicle Parts or Accessories',
    'Theft From Motor Vehicle'
]

vehicle_related = cd[cd['Crime Name2'].isin(vehicle_related_types)].copy()
vehicle_related['CrimeGroup'] = 'Vehicle-Related Crimes'
vehicle_related['Count'] = 1

agg = vehicle_related.groupby(['Year', 'City'])['Count'].sum().reset_index()
agg = agg[agg['Count'] >= 500]

vehicle_related["Place_Category"] = vehicle_related["Place"].apply(categorize_place)

place_counts = (
    vehicle_related.groupby("Place_Category")["Count"]
    .sum()
    .reset_index()
    .sort_values("Count", ascending=False)
)

fig = sp.make_subplots(
    rows=1, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Vehicle-Related Crimes by City Over Years",
        "Risk of Vehicle-Related Crimes by Place Type"
    ]
)

# Bubble chart
fig1 = px.scatter(
    agg,
    x="Year",
    y="Count",
    size="Count",
    color="City",
    hover_name="City",
    size_max=60,
    title="Vehicle-Related Crimes by City Over Years",
    color_discrete_sequence=px.colors.qualitative.Plotly_r
)
for trace in fig1.data:
    trace.legendgroup = "group1"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=1)
    
# Bar chart
fig2 = px.bar(
    place_counts,
    x="Place_Category",
    y="Count",
    color="Place_Category",
    title="Where Vehicle-Related Crimes Occur Most",
    color_discrete_sequence=px.colors.qualitative.Plotly_r
)

for trace in fig2.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=2)

# Make text readable
# fig.update_traces(textposition='middle center', textfont_size=12)

fig.update_layout(
    template='plotly_white',
    height=600,
    legend=dict(
        x=0.38,
        y=.96,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    legend2=dict(   
        x=0.85,
        y=.95,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
for i, trace in enumerate(fig.data):
    if trace.legendgroup == "group2":
        trace.legend = "legend2"
        
fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_yaxes(title_text="Crime Count", row=1, col=1)

fig.update_xaxes(title_text="Place Category", row=1, col=2)
fig.update_yaxes(title_text="Crime Count", row=1, col=2)

fig.show()

![Crime Type Chart](./images/Q5.png "a title")

# Q6 How did the 2020 pandemic and a drop in drug/weapon offences fuel the sharp decline in 'Crime Against Society' after 2019? !!Has 2 Vis
By analysing the chart in question number 2, we noticed that there is a sharp decline in Crime Against Society after 2019. Here is more clear chart for it. 

Read This (For report writer!!)
2017 the crime against society the number was `13309` and during ``2018`` it was ``13244`` which is barely a dent in statistics meanwhile during ``2019`` the crime against society was ``10922`` which is ``2000`` less where we see a pattern of decrease and ``2020`` which was peak start of covid the Crime went down by half which is ``5876`` and the years after that it kept going lower which was ``4476`` and ``3152`` for ``2021`` and ``2022`` respectively.


In [184]:
# Show Crime Name1 Crime Against Society over the years with line plot
# Line Chart for Crime Name1'Crime Against Society' types(Crime Name2) over the years top 5 types

society_crimes = cd[(cd['Crime Name1'] == 'Crime Against Society')].astype(str)
society_crime_counts = society_crimes.groupby(['Year', 'Crime Name1']).size().reset_index(name='Count')
society_type_counts = society_crimes.groupby(['Year', 'Crime Name2']).size().reset_index(name='Count')
top5_types = cd['Crime Name2'].where(cd['Crime Name1'] == 'Crime Against Society').value_counts().nlargest(5).index.tolist()
society_type_counts = society_type_counts[society_type_counts['Crime Name2'].isin(top5_types)]


fig = sp.make_subplots(
    rows=1, cols=2,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Yearly Crime Counts for Crime Against Society",
        "Yearly Crime Counts for Top 5 Crime Against Society Types"
    )
)

fig1 = px.line(
    society_crime_counts,
    x='Year',
    y='Count',
    color='Crime Name1',
    title='Yearly Crime Counts for Crime Against Society',
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Dark2
)
for trace in fig1.data:
    trace.legendgroup = "group1"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=1)


fig2 = px.line(
    society_type_counts,
    x='Year',
    y='Count',
    color='Crime Name2',
    title='Yearly Crime Counts for Top 5 Crime Against Society Types',                      
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Dark2
)
for trace in fig2.data:
    trace.legendgroup = "group2"
    trace.showlegend = True
    fig.add_trace(trace, row=1, col=2)


fig.update_layout(
    height=800,
    legend=dict(
        x=0.32,
        y=.95,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    legend2=dict(   
        x=0.85,
        y=.95,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    width=1200,
)

for i, trace in enumerate(fig.data):
    if trace.legendgroup == "group2":
        trace.legend = "legend2"
        
fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

fig.update_xaxes(title_text="Year", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.show()

![Crime Type Chart](./images/Q6.png "a title")

# Q7. Which type of crime occur more on weekend than weekdays? !!Has 2 Vis

In [171]:
df['Day_Type'] = df['Day_Name'].apply(lambda x: 'Weekend' if x in ['Saturday','Sunday'] else 'Weekday')

# --- CALCULATE WEEKDAY VS WEEKEND AVERAGES PER CRIME TYPE ---
crime_counts = (
    df.groupby(['Crime Name1', 'Day_Type'])
      .size()
      .reset_index(name='Avg_Incidents')
)
# Optional: normalize by number of unique days in each type
day_type_counts = df.groupby('Day_Type')['Date'].nunique().to_dict()
crime_counts['Avg_Incidents'] = crime_counts.apply(lambda row: row['Avg_Incidents'] / day_type_counts[row['Day_Type']], axis=1)
#remove other and not a crime
crime_counts = crime_counts[~crime_counts['Crime Name1'].isin(['Other', 'Not a Crime'])]

person_stats = (
    df[df['Crime Name1'] == 'Crime Against Person']
    .groupby('Day_Name').size()
    / df.groupby('Day_Name')['Date'].nunique()
)
person_stats = person_stats.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
person_stats = person_stats.reset_index(name='Avg_Incidents')


# --- CREATE SUBPLOTS ---
fig = sp.make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Crime Types: Weekday vs. Weekend Averages",
        "Risk of 'Crime Against Person' by Day of Week"
    ]
)

# --- ADD BAR CHART TO FIRST SUBPLOT ---
bar_fig = px.bar(
    crime_counts,
    x='Crime Name1',
    y='Avg_Incidents',
    color='Day_Type',
    barmode='group',
    color_discrete_sequence=px.colors.sequential.Viridis
)

for trace in bar_fig.data:
    trace.legendgroup = "group1"
    fig.add_trace(trace, row=1, col=1)

# --- ADD LINE CHART TO SECOND SUBPLOT ---
line_trace = go.Scatter(
    x=person_stats['Day_Name'],
    y=person_stats['Avg_Incidents'],
    mode='lines+markers',
    line=dict(width=3, color='crimson'),
    marker=dict(size=8),
    showlegend=False
)
fig.add_trace(line_trace, row=1, col=2)

# --- UPDATE AXES AND LAYOUT ---
fig.update_yaxes(title_text="Average Incidents per Day", row=1, col=1)
fig.update_yaxes(title_text="Average Incidents per Day", row=1, col=2)
fig.update_xaxes(title_text="Crime Type", row=1, col=1)
fig.update_xaxes(title_text="Day of Week", row=1, col=2)

fig.update_layout(
    legend=dict(
        x=0.23,
        y=0.95,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    height=900,
    width=900
)

fig.show()

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\1702824027.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![Crime Type Chart](./images/Q7.png "a title")

# Q8. Where does Crime against Person happen more? !!Has 2 Vis

First from the data we can see Crime against Person happens more on weekends.

In [144]:
#Classify risk categories
risk_mapping = {
    'Simple Assault': 'Assault / Physical Violence',
    'Aggravated Assault': 'Assault / Physical Violence',
    'Forcible Rape': 'Sexual Violence',
    'Fondling': 'Sexual Violence',
    'Forcible Fondling': 'Sexual Violence',
    'Forcible Sodomy': 'Sexual Violence',
    'Intimidation': 'Harassment',
    'Sexual Assault With An Object': 'Sexual Violence',
    'All Other Offenses': 'Other',
    'Murder and Nonnegligent Manslaughter': 'Homicide',
    'Kidnapping/Abduction': 'Homicide',
    'Statuory Rape': 'Sexual Violence',
    'Human Trafficking, Commercial Sex Acts': 'Human Trafficking',
    'Justifiable Homicide': 'Homicide',
    'Negligent Manslaughter': 'Homicide',
    'Human Trafficking, Involuntary Servitude': 'Human Trafficking'
}

df['Risk_Category'] = df['Crime Name2'].map(risk_mapping)
df_risk = df[df['Risk_Category'].notnull()]
df_risk['Place_Type'] = df_risk['Place'].apply(categorize_place)

#Prepare days and area for visualizations
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'] 
top_districts = ['MONTGOMERY VILLAGE', 'WHEATON', 'SILVER SPRING', 'ROCKVILLE', 'GERMANTOWN'] 
heatmap_data = df_risk[df_risk['Police District Name'] == 'MONTGOMERY VILLAGE'].groupby(['Day_Name', 'Hour']).size().unstack(fill_value=0).reindex(days_order) 
loc_counts = df_risk[df_risk['Police District Name'].isin(top_districts)].groupby(['Police District Name', 'Place_Type']).size().reset_index(name='Count') 
comp_counts = df_risk[df_risk['Police District Name'].isin(top_districts)].groupby(['Police District Name', 'Risk_Category']).size().unstack(fill_value=0)
df_risk = df_risk[df_risk['Risk_Category'] != 'Other']
df_risk = df_risk[df_risk['Police District Name'] != 'OTHER']

fig = sp.make_subplots( rows=2, cols=2, subplot_titles=[ 
    'Crime Against Person Risk Treemap', 
    'Where is it Happening?', 'Risk Composition by District', "",
     ], 
    specs=[ [{"colspan": 2,"type": "treemap"}, None], 
           [{"type": "bar"}, {"type": "bar"}] ] ) 


#fig1 Treemap
treemap_counts = df_risk.groupby(['Police District Name', 'Risk_Category']).size().reset_index(name='Count')

treemap_counts['Percentage'] = treemap_counts.groupby('Police District Name')['Count'].transform(lambda x: (x / x.sum()) * 100)
treemap_counts['Label'] = treemap_counts.apply(lambda row: f"{row['Risk_Category']} – {row['Percentage']:.1f}%", axis=1)

fig1 = px.treemap(
    treemap_counts,
    path=['Police District Name', 'Risk_Category'],  
    values='Count',
    color='Police District Name',
    color_discrete_sequence=px.colors.qualitative.Set3,
)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

place_counts = df_risk.groupby(['Police District Name', 'Place_Type']).size().reset_index(name='Count')
fig2 = px.bar( 
    place_counts, 
    x="Police District Name", 
    y="Count", color="Place_Type", 
    barmode="group", color_discrete_sequence=px.colors.qualitative.Plotly ) 


top3_comp = df_risk.groupby(['Police District Name', 'Risk_Category']).size().reset_index(name='Count') #remove OTHER
top3_comp = top3_comp.sort_values(['Police District Name', 'Count'], ascending=[True, False])
top3_comp = top3_comp.groupby('Police District Name').head(3)
fig3 = px.bar( 
    top3_comp,
    x="Police District Name", 
    y="Count", 
    color="Risk_Category",
    barmode="stack", color_discrete_sequence=["#e74c3c", "#8e44ad", "#f1c40f"] ) 


for trace in fig2.data: 
    trace.legendgroup = "group1" 
    fig.add_trace(trace, row=2, col=1) 
for trace in fig3.data: 
    trace.legendgroup = "group2" 
    fig.add_trace(trace, row=2, col=2) 

fig.update_layout( 
    legend=dict( x=0.32, y=.19, bgcolor="rgba(255,255,255,0.7)" ), 
    legend2=dict( x=.84, y=.19, bgcolor="rgba(255,255,255,0.7)" ), 
    height=1000,
    width=1400
    ) 
num_treemap_traces = len(fig1.data)
for i, trace in enumerate(fig.data[num_treemap_traces:]):
    if trace.legendgroup == "group2":
        trace.legend = "legend2"

fig.update_xaxes(title_text="Police Districts", row=2, col=1)
fig.update_yaxes(title_text="Incident Count", row=2, col=1)
fig.update_xaxes(title_text="Police Districts", row=2, col=2) 
fig.update_yaxes(title_text="Incident Count", row=2, col=2)
fig.show()

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\3589629772.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



![Crime Type Chart](./images/Q8.png "a title")

# Q9. What is Severity and Response time for each police districts and safest area in Montgomery to live in? !!Has 2 Vis

In [181]:
main_crimes = ['Crime Against Property', 'Crime Against Society', 'Crime Against Person']
df = df[df['Police District Name'] != 'OTHER']
df = df[df['Crime Name1'].isin(main_crimes)]

# --- 2. Compute Severity Score ---
severity_score_map = {
    'Crime Against Person': 10,
    'Crime Against Society': 5,
    'Crime Against Property': 1
}
df['Severity_Score'] = df['Crime Name1'].map(severity_score_map)



district_stats = (
    df.groupby('Police District Name')['Severity_Score']
    .agg(['mean', 'count', 'std'])
    .reset_index()
    .sort_values('mean')  # safest first
)

df['response_time_min'] = (df['Dispatch Date / Time'] - df['Start_Date_Time']).dt.total_seconds() / 60
df = df[(df['response_time_min'] > 0) & (df['response_time_min'] <= 120)]

severity_label_map = {
    'Crime Against Person': 'High',
    'Crime Against Society': 'Medium',
    'Crime Against Property': 'Low'
}
df['severity'] = df['Crime Name1'].map(severity_label_map)
df = df.dropna(subset=['severity'])

district_summary = (
    df.groupby(['Police District Name', 'severity'])['response_time_min']
    .mean()
    .reset_index()
)

fig = sp.make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.12,
    subplot_titles=(
        "Average Response Time by District and Severity",
        "Safest Area in Montgomery County"
    )
)

severity_colors = {"High": "#440154", "Medium": "#31688e", "Low": "#35b779"}
for sev in ['High', 'Medium', 'Low']:
    df_sev = district_summary[district_summary['severity'] == sev]
    fig.add_trace(
        go.Bar(
            x=df_sev['response_time_min'],
            y=df_sev['Police District Name'],
            name=sev,
            orientation='h',
            marker_color=severity_colors[sev],
        ),
        row=1, col=1
    )

fig.add_trace(
    go.Bar(
        x=district_stats['mean'],
        y=district_stats['Police District Name'],
        text=district_stats['Police District Name'],  # Show district names on bars
        textposition='inside',
        marker=dict(color=district_stats['mean'], colorscale='Viridis'),
        name='Avg Severity Score',
        orientation='h'
    ),
    row=1, col=2
)

# --- 11. Update Layout ---
fig.update_layout(
    legend=dict(
        x=0.32,
        y=.96,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    barmode='group',
    legend_title="Severity / Score",
    yaxis=dict(title="Police District"),
    yaxis2=dict(title="Police District")
)
fig.update_xaxes(title_text="Average Response Time (minutes)", row=1, col=1)
fig.update_xaxes(title_text="Average Crime Severity Score<br>(1=Property, 5=Society, 10=Person)", row=1, col=2)


fig.show()

![Crime Type Chart](./images/Q9.png "a title")

# Q10. What are the most occuring crime in each area and count of each respective crime? also include place type

In [256]:
filtered = cd[cd['Police District Name'] != 'OTHER'].copy()
filtered = filtered[filtered['Crime Name1'].isin(main_crimes)]

# Group by district, crime, place
grouped_tm = filtered.groupby(['Police District Name', 'Crime Name1', 'Place_Category']).size().reset_index(name='Count')

# Get the most frequent crime per district
idx = grouped_tm.groupby('Police District Name')['Count'].idxmax()
top_crimes_per_district = grouped_tm.loc[idx].reset_index(drop=True)

total_per_district = filtered.groupby('Police District Name').size().reset_index(name='Total')
top_crimes_per_district = top_crimes_per_district.merge(total_per_district, on='Police District Name')
top_crimes_per_district['Percentage'] = (top_crimes_per_district['Count'] / top_crimes_per_district['Total'] * 100).round(2)

# Add percentage and label
top_crimes_per_district['label_text'] = (
    top_crimes_per_district['Crime Name1'].astype(str) + '<br>' +
    'Count: ' + top_crimes_per_district['Count'].astype(str) + '<br>' +
    'Place: ' + top_crimes_per_district['Place_Category'].astype(str) + '<br>' +
    'Percentage: ' + top_crimes_per_district['Percentage'].astype(str) + '%'
)

# --- Plot 1: Treemap Figure Creation (just for traces) ---
fig_treemap_px = px.treemap(
    top_crimes_per_district,
    path=['Police District Name', 'Crime Name1'],
    values='Count',
    color='Count',
    color_continuous_scale='Reds',

    hover_data={}
)
fig_treemap_px.data[0].text = top_crimes_per_district['label_text']
fig_treemap_px.data[0].texttemplate = "%{text}"
fig_treemap_px.data[0].textfont = dict(size=12)
fig_treemap_px.data[0].textposition = "middle center"
fig_treemap_px.data[0].textinfo = "text"


# --- Plot 2: Sunburst Data Prep ---
filtered_sb = cd[cd['Police District Name'] != 'OTHER'].copy()
grouped_sb = filtered_sb.groupby(['Police District Name', 'Crime Name1', 'Place_Category']).size().reset_index(name='Count')
grouped_sb['Rank'] = grouped_sb.groupby('Police District Name')['Count'].rank(method='dense', ascending=False)
top_3_crimes_per_district = grouped_sb[grouped_sb['Rank'] <= 3].copy()

# --- Plot 2: Sunburst Figure Creation (just for traces) ---
fig_sunburst_px = px.sunburst(
    top_3_crimes_per_district,
    path=['Police District Name', 'Crime Name1', 'Place_Category'],
    values='Count',
    color='Count',
    color_continuous_scale='Reds',
)
fig_sunburst_px.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage of Parent: %{percentParent}<extra></extra>'
)

# --- Combine Plots using Subplots ---
# Create a subplot figure with 1 row and 2 columns
# Treemaps and Sunbursts don't use standard axes, so the 'specs' must be set
# to [{'type': 'domain'}] for each subplot
fig = sp.make_subplots(
    rows=1, 
    cols=2, 
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=(
        'Most Occurring Crime per District', 
        'Top 3 Most Occurring Crimes per District'
    )
)

# Add the Treemap trace (first trace from the Treemap PX figure)
fig.add_trace(
    fig_treemap_px.data[0], 
    row=1, 
    col=1
)

# Add the Sunburst trace (first trace from the Sunburst PX figure)
fig.add_trace(
    fig_sunburst_px.data[0], 
    row=1, 
    col=2
)

# Update layout for the combined figure
fig.update_layout(
    template='plotly_white',
    height=800,
    width=1600,
    margin=dict(t=100, l=50, r=50, b=50),
    coloraxis=dict(colorscale='Reds'),  
    title_text="Crime Patterns by Police District",
    title_x=0.5
)

fig.show()

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\74306033.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Willow7T\AppData\Local\Temp\ipykernel_37316\74306033.py:42: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



![Crime Type Chart](./images/Q10.png "a title")